# 31. Attribute lookup, introspection, and dynamic access

**Level:** Advanced

**Learning outcome:** Use safe introspection and understand missing-attribute fallback without hiding bugs.

**Prerequisites:** 03, 12, and 21: properties, special methods, and descriptors.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](30_Iterators_Generators_and_Lazy_Object_APIs.ipynb) | [Next notebook](32_Metaclasses_and_Class_Creation.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Mental model
getattr, setattr, hasattr, vars, and dir help inspect objects, but inspection may invoke behavior.
For example, reading a property can run validation or perform work. hasattr performs attribute
lookup and treats AttributeError as absence; it is not a harmless dictionary membership test.
dir lists useful names but is not a complete schema or authorization policy.

In ordinary lookup, data descriptors can take precedence over instance storage. Non-data
descriptors and ordinary class attributes have different precedence. A property with a setter
is a data descriptor. Functions on classes use descriptor behavior to become bound methods.
Use the descriptor notebook for reusable field validation.

__getattr__ is called as a fallback when normal lookup fails. It must raise AttributeError for
names it cannot provide. Returning None for every unknown attribute hides spelling mistakes.
__getattribute__ intercepts ordinary access more broadly; if you use it, delegate through
object.__getattribute__ to avoid calling yourself recursively. Most automation does not need it.

Dynamic input should not select unrestricted attribute names. Use an explicit allowlist or a
mapping of supported operations. Introspection is not a security boundary and underscores do
not make data inaccessible.

## Worked example 1: Allowlisted dynamic updates still validate

**Problem:** Apply supported configuration fields without allowing arbitrary attribute writes.

### Step-by-step explanation
1. A property establishes the same validation contract as a normal assignment.
2. The update method restricts which public field a caller may select.
3. setattr uses normal attribute assignment and therefore invokes the setter.
4. Reject an internal backing-field name before it can bypass the public interface.

**Expected output:**
```text
internal-field update rejected
['_mtu']
```

**Common mistake / limitation:** This application allowlist is not isolation from code already able to access the object directly.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [1]:
class Interface:
    def __init__(self):
        self.mtu = 1500

    @property
    def mtu(self):
        return self._mtu

    @mtu.setter
    def mtu(self, value):
        if type(value) is not int or not 576 <= value <= 9216:
            raise ValueError("invalid MTU")
        self._mtu = value

    def update(self, field, value):
        if field != "mtu":
            raise ValueError("unsupported field")
        setattr(self, field, value)

interface = Interface()
interface.update("mtu", 9000)
assert getattr(interface, "mtu") == 9000
try:
    interface.update("_mtu", 1)
except ValueError:
    assert interface.mtu == 9000
    print("internal-field update rejected")
else:
    raise AssertionError("unsupported field accepted")
print(sorted(vars(interface)))

internal-field update rejected
['_mtu']


### Practice 1

Add a second allowed property and keep invalid updates atomic per field.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Worked example 2: Missing-attribute fallback preserves real errors

**Problem:** A compatibility alias can be explicit without accepting arbitrary typos.

### Step-by-step explanation
1. Normal hostname lookup succeeds without the fallback.
2. Only the documented legacy alias is synthesized.
3. Unknown names raise the correct exception type.
4. Demonstrate how hasattr observes the same presence/absence contract.

**Expected output:**
```text
unknown attribute reported
```

**Common mistake / limitation:** Do not turn every unknown attribute into None or you will hide genuine programming mistakes.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [2]:
class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def __getattr__(self, name):
        if name == "legacy_name":
            return self.hostname
        raise AttributeError(f"{type(self).__name__} has no attribute {name!r}")

device = Device("edge-01")
assert device.legacy_name == "edge-01"
assert hasattr(device, "hostname")
assert not hasattr(device, "hostnmae")
try:
    device.hostnmae
except AttributeError as error:
    assert "hostnmae" in str(error)
    print("unknown attribute reported")
else:
    raise AssertionError("typo hidden")

unknown attribute reported


### Practice 2

Add a deprecation policy for the alias without changing the canonical field.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Worked example 3: A narrow __getattribute__ trace

**Problem:** Trace one public access while avoiding recursive interception.

### Step-by-step explanation
1. Intercept only the hostname name for this small demonstration.
2. Fetch reads through object.__getattribute__, not self.reads inside the interceptor.
3. Delegate the requested lookup through the base implementation.
4. Show that explicitly using the base operation bypasses this custom trace.

**Expected output:**
```text
['hostname']
```

**Common mistake / limitation:** Special method lookup for language syntax may bypass ordinary instance interception; do not assume this traces everything.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [3]:
class TracedDevice:
    def __init__(self, hostname):
        self.hostname = hostname
        self.reads = []

    def __getattribute__(self, name):
        if name == "hostname":
            reads = object.__getattribute__(self, "reads")
            reads.append(name)
        return object.__getattribute__(self, name)

device = TracedDevice("edge-01")
assert device.hostname == "edge-01"
assert device.reads == ["hostname"]
assert object.__getattribute__(device, "hostname") == "edge-01"
assert device.reads == ["hostname"]
print(device.reads)

['hostname']


### Practice 3

Replace interception with a property-based trace and explain why it is simpler for one field.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Check your understanding: Attribute lookup, introspection, and dynamic access

**You should now be able to:** Use safe introspection and understand missing-attribute fallback without hiding bugs.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/reference/datamodel.html`
- `https://docs.python.org/3.13/library/functions.html`

References describe Python contracts. They do not certify any vendor command or network policy.